# OCT Binary Classification — VGG16 Method 1: Base Model (Transfer Learning)

**Approach:** Frozen VGG16 pretrained on ImageNet. Train only the classifier head.
No data augmentation, no fine-tuning. Pickle checkpointing for Kaggle session recovery.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, pickle
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Data Loading

In [ ]:
# Dataset paths
train_dir = '/kaggle/input/datasets/mohamedaminedrif/oct-binary/train'
val_dir = '/kaggle/input/datasets/mohamedaminedrif/oct-binary/val'
test_dir = '/kaggle/input/datasets/mohamedaminedrif/oct-binary/test'

for name, path in [('Train', train_dir), ('Validation', val_dir), ('Test', test_dir)]:
    if os.path.exists(path):
        print(f'\n{name} set:')
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                print(f'  {cls}: {len(os.listdir(cls_path))} images')

## Data Generators (No Augmentation)

In [ ]:
# Data generators — rescaling ONLY, no augmentation
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = datagen.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True)
val_gen = datagen.flow_from_directory(val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)
test_gen = datagen.flow_from_directory(test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)

print(f'Class mapping: {train_gen.class_indices}')
print(f'Training: {train_gen.samples} | Validation: {val_gen.samples} | Test: {test_gen.samples}')

## Build VGG16 Model (Frozen Base)

In [ ]:
# Build model with frozen base
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(256, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## Training with Pickle Checkpointing

In [ ]:
# Training with pickle checkpointing
EPOCHS = 15
CKPT = '/kaggle/working/oct_vgg16_m1_ckpt.pkl'
WGTS = '/kaggle/working/oct_vgg16_m1_weights.h5'

hist = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
start = 0

if os.path.exists(CKPT):
    with open(CKPT, 'rb') as f:
        ckpt = pickle.load(f)
    model.load_weights(WGTS)
    start = ckpt['epoch']
    hist = ckpt['history']
    print(f'Resuming from epoch {start}')

for epoch in range(start, EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    h = model.fit(train_gen, epochs=epoch+1, initial_epoch=epoch, validation_data=val_gen)
    for k in hist:
        hist[k].extend(h.history[k])
    model.save_weights(WGTS)
    with open(CKPT, 'wb') as f:
        pickle.dump({'epoch': epoch+1, 'history': hist}, f)

print(f'\nBest Val Accuracy: {max(hist["val_accuracy"])*100:.2f}%')

## Training Curves

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(hist['accuracy'], label='Train'); ax1.plot(hist['val_accuracy'], label='Val')
ax1.set_title('Accuracy — VGG16 Base Model', fontweight='bold'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(hist['loss'], label='Train'); ax2.plot(hist['val_loss'], label='Val')
ax2.set_title('Loss — VGG16 Base Model', fontweight='bold'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Evaluation

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(test_gen)
print(f'\nTest Accuracy: {test_acc*100:.2f}%')
print(f'Test Loss: {test_loss:.4f}')

# Confusion Matrix
test_gen.reset()
y_pred = (model.predict(test_gen) > 0.5).astype(int).flatten()
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, annot_kws={'size': 16})
plt.title('Confusion Matrix — VGG16 (Base Model)', fontsize=16, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

## Export Results

In [ ]:
import csv, os, json
from sklearn.metrics import classification_report

# ── Save confusion matrix as PNG ──
os.makedirs('/kaggle/working/results', exist_ok=True)
cm_path = '/kaggle/working/results/oct_vgg16_m1_confusion_matrix.png'
fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={'size': 16}, ax=ax_cm)
ax_cm.set_title('Confusion Matrix — oct_vgg16_m1', fontsize=14, fontweight='bold')
ax_cm.set_xlabel('Predicted'); ax_cm.set_ylabel('True')
fig_cm.tight_layout()
fig_cm.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close(fig_cm)
print(f'Confusion matrix saved → {cm_path}')

# ── Save metrics to CSV ──
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
csv_path = '/kaggle/working/results/oct_vgg16_m1_results.csv'
rows = []
for label, metrics in report_dict.items():
    if isinstance(metrics, dict):
        rows.append({
            'model':     'oct_vgg16_m1',
            'class':     label,
            'precision': round(metrics['precision'], 4),
            'recall':    round(metrics['recall'],    4),
            'f1-score':  round(metrics['f1-score'],  4),
            'support':   metrics['support'],
            'test_acc':  round(test_acc, 4),
            'test_loss': round(test_loss, 4),
        })

write_header = not os.path.exists(csv_path)
with open(csv_path, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    if write_header:
        writer.writeheader()
    writer.writerows(rows)

print(f'Results saved → {csv_path}')
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')


## Prediction Visualization

In [ ]:
# Visualize predictions
test_gen.reset()
images, labels = next(test_gen)
predictions = model.predict(images)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    true_label = class_names[int(labels[i])]
    pred_label = class_names[int(predictions[i] > 0.5)]
    confidence = predictions[i][0] if predictions[i] > 0.5 else 1 - predictions[i][0]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1%})', fontsize=10, color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle('VGG16 — Base Model Predictions', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

## Save Model

In [ ]:
model.save('vgg16_method1_base.keras')
print('Model saved as vgg16_method1_base.keras')